In [1]:
# DeepSeek system prompt for GRPO based training
SYSTEM_PROMPT = (
    "A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant "
    "first thinks about the reasoning process in the mind and then provides the user with the answer. The reasoning "
    "process and answer are enclosed within <think> </think> and <answer> </answer> tags, respectively, i.e., "
    "<think> reasoning process here </think><answer> answer here </answer>"
)

In [2]:
!pip install datasets torc transformers
!pip install --upgrade accelerate




Looking in indexes: https://mirrors.tencent.com/pypi/simple/
Looking in indexes: https://mirrors.tencent.com/pypi/simple/
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.6.0
    Uninstalling accelerate-1.6.0:
      Successfully uninstalled accelerate-1.6.0


In [3]:
from dataclasses import dataclass
from datasets import load_dataset  # 添加这行导入

/root/.pyenv/versions/3.11.1/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:

# 构造训练数据的函数
def make_conversation(example):
    """
    将数据集中的示例转换为对话格式。
    
    参数:
        example: 数据集中的单个示例
    
    返回:
        转换后的对话格式数据
    """
    """将数据集示例转换为对话格式。"""
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": example["problem"]},
        ],
    }

In [5]:
# 加载并准备数据集
def load_math_dataset():
    """加载并准备数学数据集。"""
    dataset = load_dataset(
        "AI-MO/NuminaMath-TIR",
        name="default",
        split=['train', 'test']
    )
    
    # 将拆分转换为字典
    dataset = {
        'train': dataset[0],
        'test': dataset[1]
    }
    
    # 应用对话格式
    for split in dataset:
        dataset[split] = dataset[split].map(make_conversation)

        # 如果存在“messages”列，则删除
        if "messages" in dataset[split].column_names:
            dataset[split] = dataset[split].remove_columns("messages")
    
    return dataset

In [6]:
#加载我们的训练数据集并打印训练/测试大小
dataset = load_math_dataset()

print(f"Train set size: {len(dataset['train'])}")
print(f"Test set size: {len(dataset['test'])}")

Train set size: 72441
Test set size: 99


In [7]:
def validate_dataset(dataset):
    """对数据集执行基本的验证检查。"""
    
    # 定义数据集所需的字段
    required_fields = ["problem", "prompt"]

    # 遍历数据集的 'train' 和 'test' 划分
    for split in ['train', 'test']:
        print(f"\n正在验证 {split} 划分:")

        # 从数据集中获取列名
        fields = dataset[split].column_names

        # 检查是否有任何必需的字段缺失
        missing = [field for field in required_fields if field not in fields]
        if missing:
            print(f"警告: 缺失的字段: {missing}")  # 若字段缺失则发出警告
        else:
            print("✓ 所有必需字段均存在")  # 确认所有字段都存在

        # 从数据集划分中获取第一个样本
        sample = dataset[split][0]

        # 提取 'prompt' 字段，该字段包含一个消息列表
        messages = sample['prompt']

        # 验证提示格式：
        # - 它应该至少包含两条消息
        # - 第一条消息应该来自 'system' 角色
        # - 第二条消息应该来自 'user' 角色
        if (len(messages) >= 2 and
            messages[0]['role'] == 'system' and
            messages[1]['role'] == 'user'):
            print("✓ 提示格式正确")  # 确认格式正确
        else:
            print("警告: 提示格式不正确")  # 若格式不正确则发出警告

# 验证数据集
validate_dataset(dataset)


正在验证 train 划分:
✓ 所有必需字段均存在
✓ 提示格式正确

正在验证 test 划分:
✓ 所有必需字段均存在
✓ 提示格式正确


In [8]:
# 定义用于奖励函数参数的GRPOScriptArguments类
from dataclasses import dataclass, field

@dataclass
class GRPOScriptArguments:
    """
    用于GRPO训练的脚本参数，具体与奖励函数相关。
    """

    # 奖励函数列表，默认包含 "accuracy" 和 "format"
    reward_funcs: list[str] = field(
        default_factory=lambda: ["accuracy", "format"],
        metadata={
            # 帮助信息，说明可能的奖励函数值
            "help": "奖励函数列表。可能的值: 'accuracy', 'format', 'reasoning_steps', 'cosine', 'repetition_penalty'"
        },
    )
    # 错误答案余弦缩放的最小奖励值
    cosine_min_value_wrong: float = field(
        default=-0.5,
        metadata={"help": "错误答案余弦缩放的最小奖励值"}
    )
    # 错误答案余弦缩放的最大奖励值
    cosine_max_value_wrong: float = field(
        default=-0.1,
        metadata={"help": "错误答案余弦缩放的最大奖励值"}
    )
    # 正确答案余弦缩放的最小奖励值
    cosine_min_value_correct: float = field(
        default=0.8,
        metadata={"help": "正确答案余弦缩放的最小奖励值"}
    )
    # 正确答案余弦缩放的最大奖励值
    cosine_max_value_correct: float = field(
        default=1.0,
        metadata={"help": "正确答案余弦缩放的最大奖励值"}
    )
    # 余弦缩放的最大长度
    cosine_max_len: int = field(
        default=1000,
        metadata={"help": "余弦缩放的最大长度"}
    )
    # 重复惩罚奖励的n-gram数量
    repetition_n_grams: int = field(
        default=3,
        metadata={"help": "重复惩罚奖励的n-gram数量"}
    )
    # 重复惩罚奖励的最大（负）惩罚值
    repetition_max_penalty: float = field(
        default=-0.1,
        metadata={"help": "重复惩罚奖励的最大（负）惩罚值"}
    )

In [9]:
# 从transformers库中定义TrainingArguments类的实例
from transformers import TrainingArguments

# 假设OUTPUT_DIR已经定义，例如：
OUTPUT_DIR = "results"

# 定义训练参数
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    dataloader_num_workers=2,
    seed=42,
    bf16=True,
    report_to="none"  # 如果需要报告训练结果，可以使用这个参数
)



In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from typing import Optional
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
@dataclass
class ModelConfig:
    """
    模型的配置类。
    """
    # 预训练模型的路径或者来自huggingface.co/models的模型标识符
    model_name_or_path: str = field(
        default=MODEL_NAME, 
        metadata={"help": "预训练模型的路径或者来自huggingface.co/models的模型标识符"}
    )
    # 要使用的具体模型版本（可以是分支名、标签名或提交ID）
    model_revision: Optional[str] = field(
        default="main", 
        metadata={"help": "要使用的具体模型版本（可以是分支名、标签名或提交ID）"}
    )
    # 覆盖默认的 `torch_dtype`，并在此数据类型下加载模型
    torch_dtype: Optional[str] = field(
        default="bfloat16", 
        metadata={"help": "覆盖默认的 `torch_dtype`，并在此数据类型下加载模型"}
    )
    # 加载模型和分词器时是否信任远程代码
    trust_remote_code: bool = field(
        default=True, 
        metadata={"help": "加载模型和分词器时是否信任远程代码"}
    )
    # 要使用的注意力机制实现方式，可选 'flash_attention_2' 或 None
    attn_implementation: Optional[str] = field(
        default="flash_attention_2", 
        metadata={"help": "要使用的注意力机制实现方式。可选 'flash_attention_2' 或 None"}
    )

In [11]:
# 实例化配置对象
# 实例化GRPOScriptArguments类，得到包含奖励函数相关参数配置的对象
script_args = GRPOScriptArguments()
# 实例化ModelConfig类，得到包含模型相关参数配置的对象
model_args = ModelConfig()

In [12]:
# 这里假设这样去定义，请根据自己的需求去定义
def accuracy_reward(completions, solution, **kwargs):
    """
    Reward function to check if the model's response is mathematically 
    equivalent to the ground truth solution.
    Uses latex2sympy2 for parsing and math_verify for validation.
    """
    
# 提取响应
    contents = [completion[0]["content"] for completion in completions]
    rewards = []
    
    for content, sol in zip(contents, solution):
# 解析真实答案
        gold_parsed = parse(sol, extraction_mode="first_match", 
                            extraction_config=[LatexExtractionConfig()])
        
        if gold_parsed:  # 检查解析是否成功
# 使用宽松的正则化方法解析模型的答案
            answer_parsed = parse(
                content,
                extraction_config=[
                    LatexExtractionConfig(
                        normalization_config=NormalizationConfig(
                            nits=False,
                            malformed_operators=False,
                            basic_latex=True,
                            equations=True,
                            boxed="all",
                            units=True,
                        ),
                        boxed_match_priority=0,
                        try_extract_without_anchor=False,
                    )
                ],
                extraction_mode="first_match",
            )

# 正确奖励 1.0，错误奖励 0.0
            reward = float(verify(answer_parsed, gold_parsed))
        else:
# 如果无法解析基本事实，则分配中性奖励（0.5）
            reward = 0.5
            print("Warning: Failed to parse gold solution:", sol)

        rewards.append(reward)
    
    return rewards

In [13]:
# 这里假设这样去定义，请根据自己的需求去定义
def format_reward(completions, **kwargs):
    """
    奖励函数，用于检查模型输出是否具有正确的格式，如 <think>...</think> 和 <answer>...</answer>。
    """
    # 定义正则表达式模式以匹配所需的格式
    pattern = r"^<think>.*?</think>\s*<answer>.*?</answer>$"
    # 提取每个完成项的内容
    rewards = []
    for completion in completions:
        content = completion[0]["content"]
        # 使用正则表达式匹配模式
        if re.match(pattern, content, re.DOTALL | re.MULTILINE):
            rewards.append(1.0)  # 如果格式正确，则奖励为 1.0
        else:
            rewards.append(0.0)  # 如果格式错误，则奖励为 0.0
    return rewards

In [14]:
# 这里假设这样去定义，请根据自己的需求去定义
import re

def reasoning_steps_reward(completions, **kwargs):
    """
    奖励函数，用于鼓励模型展示清晰的推理步骤。
    """
    # 定义一个正则表达式模式，用于匹配推理步骤的指示符
    pattern = r"Step \d+:|^\d+\.|\n-|\n\*|First,|Second,|Next,|Finally,"
    # 提取每个完成项的内容
    completion_contents = [completion[0]["content"] for completion in completions]
    # 计算每个完成项中推理步骤指示符的数量
    matches = [len(re.findall(pattern, content, re.MULTILINE)) for content in completion_contents]
    # 奖励与推理步骤数量成正比，最大值为1.0
    # 这里使用了一个“魔法数字”3 - 鼓励至少3个步骤以获得满分
    return [min(1.0, count / 3) for count in matches]


In [15]:
# 这里假设这样去定义，请根据自己的需求去定义
import math

def get_cosine_scaled_reward(min_value_wrong: float = -0.5,
                             max_value_wrong: float = -0.1,
                             min_value_correct: float = 0.8,
                             max_value_correct: float = 1.0,
                             max_len: int = 1000):
    """
    返回一个基于答案长度缩放准确率奖励的函数。
    正确答案会得到更短、更直接的解答更高的奖励，错误答案则相反。
    
    参数:
        min_value_wrong (float): 错误答案的最小奖励值，默认为 -0.5。
        max_value_wrong (float): 错误答案的最大奖励值，默认为 -0.1。
        min_value_correct (float): 正确答案的最小奖励值，默认为 0.8。
        max_value_correct (float): 正确答案的最大奖励值，默认为 1.0。
        max_len (int): 用于缩放的最大长度，默认为 1000。
    
    返回:
        function: 接收生成内容、解决方案和准确率奖励，返回缩放后的奖励列表。
    """
    def cosine_scaled_reward(completions, solution, accuracy_rewards, **kwargs):
        """
        基于答案长度缩放准确率奖励的函数。
        
        参数:
            completions (list): 模型的生成内容列表。
            solution: 正确答案，用于计算准确率奖励。
            accuracy_rewards (list): 准确率奖励列表。
            **kwargs: 其他关键字参数。
        
        返回:
            list: 缩放后的奖励列表。
        """
        contents = [completion[0]["content"] for completion in completions]
        rewards = []
        for content, acc_reward in zip(contents, accuracy_rewards):
            gen_len = len(content)  # 生成内容的长度
            progress = gen_len / max_len  # 生成内容长度与最大长度的比例
            cosine = math.cos(progress * math.pi)  # 根据进度计算的余弦值
            if acc_reward > 0.5:  # 假设准确率奖励大于 0.5 表示正确答案
                min_value = min_value_correct
                max_value = max_value_correct
            else:  # 错误答案
                min_value = max_value_wrong  # 注意这里交换了最小值和最大值
                max_value = min_value_wrong
            # 使用余弦缩放公式计算奖励
            reward = min_value + 0.5 * (max_value - min_value) * (1.0 + cosine)
            rewards.append(float(reward))
        return rewards
    return cosine_scaled_reward


In [16]:
# 这里假设这样去定义，请根据自己的需求去定义
def reasoning_steps_reward(completions, **kwargs):
    """
    奖励函数，用于鼓励模型展示清晰的推理步骤。
    """
    # 定义一个正则表达式模式，用于匹配推理步骤的指示符
    pattern = r"Step \d+:|^\d+\.|\n-|\n\*|First,|Second,|Next,|Finally,"
    # 提取每个完成项的内容
    completion_contents = [completion[0]["content"] for completion in completions]
    # 计算每个完成项中推理步骤指示符的数量
    matches = [len(re.findall(pattern, content, re.MULTILINE)) for content in completion_contents]
    # 奖励与推理步骤数量成正比，最大值为1.0
    # 这里使用了一个“魔法数字”3 - 鼓励至少3个步骤以获得满分
    return [min(1.0, count / 3) for count in matches]


In [17]:
# 这里假设这样去定义，请根据自己的需求去定义
def get_repetition_penalty_reward(ngram_size: int = 3, max_penalty: float = -0.1):
    """
    Returns a repetition penalty reward function. Penalizes repetitions of n-grams
    in the generated text.
    """
    if max_penalty > 0:
        raise ValueError(f"max_penalty {max_penalty} should not be positive")
    def zipngram(text: str, ngram_size: int):
        """Helper function to generate n-grams from text."""
        words = text.lower().split() # Lowercase and split into words
        return zip(*[words[i:] for i in range(ngram_size)]) # Create n-grams
    def repetition_penalty_reward(completions, **kwargs) -> float:
        """
        Repetition penalty reward function.
        """
        contents = [completion[0]["content"] for completion in completions]
        rewards = []
        for completion in contents:
            if completion == "": # No penalty for empty completions
                rewards.append(0.0)
                continue
            if len(completion.split()) < ngram_size: # No penalty for short completions
                rewards.append(0.0)
                continue
            ngrams = set() # Use a set to store unique n-grams
            total = 0
            for ng in zipngram(completion, ngram_size): # Generate n-grams
                ngrams.add(ng) # Add n-gram to the set (duplicates are ignored)
                total += 1 # Count total n-grams
            # Calculate scaling factor: more repetition -> higher scaling
            scaling = 1 - len(ngrams) / total
            reward = scaling * max_penalty # Apply penalty based on scaling
            rewards.append(reward)
        return rewards
    return repetition_penalty_reward


In [18]:
# 基于脚本参数获取奖励函数的工具函数
def get_reward_functions(script_args):
    """
    根据脚本参数返回一个奖励函数列表。
    """
    # 用于存储奖励函数的列表
    reward_funcs_list = []
    # 奖励函数注册表，将奖励函数名称映射到具体的奖励函数
    reward_funcs_registry = {
        # 假设 accuracy_reward 函数在之前的步骤中已经定义
        "accuracy": accuracy_reward,
        # 假设 format_reward 函数在之前的步骤中已经定义
        "format": format_reward,
        # 假设 reasoning_steps_reward 函数已经定义
        "reasoning_steps": reasoning_steps_reward,
        # 假设 get_cosine_scaled_reward 函数已经定义
        "cosine": get_cosine_scaled_reward(
            # 从脚本参数中获取错误答案余弦缩放的最小奖励值
            min_value_wrong=script_args.cosine_min_value_wrong,
            # 从脚本参数中获取错误答案余弦缩放的最大奖励值
            max_value_wrong=script_args.cosine_max_value_wrong,
            # 从脚本参数中获取正确答案余弦缩放的最小奖励值
            min_value_correct=script_args.cosine_min_value_correct,
            # 从脚本参数中获取正确答案余弦缩放的最大奖励值
            max_value_correct=script_args.cosine_max_value_correct,
            # 从脚本参数中获取余弦缩放的最大长度
            max_len=script_args.cosine_max_len,
        ),
        # 假设 get_repetition_penalty_reward 函数已经定义
        "repetition_penalty": get_repetition_penalty_reward(
            # 从脚本参数中获取重复惩罚奖励的 n-gram 数量
            ngram_size=script_args.repetition_n_grams,
            # 从脚本参数中获取重复惩罚奖励的最大（负）惩罚值
            max_penalty=script_args.repetition_max_penalty,
        ),
    }

    # 遍历脚本参数中指定的奖励函数名称
    for func_name in script_args.reward_funcs:
        # 检查指定的奖励函数名称是否在注册表中
        if func_name not in reward_funcs_registry:
            # 如果不在注册表中，抛出值错误
            raise ValueError(f"奖励函数 '{func_name}' 未在注册表中找到。")
        # 将对应的奖励函数添加到奖励函数列表中
        reward_funcs_list.append(reward_funcs_registry[func_name])

    # 返回奖励函数列表
    return reward_funcs_list

In [19]:
import logging
from transformers import TrainerCallback , TrainerState,  TrainerControl

# 获取当前模块的日志记录器
logger = logging.getLogger(__name__)

class LoggingCallback(TrainerCallback):
    """
    一个简单的回调类，用于在特定步骤记录训练信息。
    """
    def on_step_end(self, args: TrainingArguments, state: TrainerState, control: TrainerControl, **kwargs):
        # 检查当前全局步数是否是日志记录步数的整数倍
        if state.global_step % args.logging_steps == 0:
            # 从日志历史记录中获取最新的损失值，如果没有则为 None
            loss = state.log_history[-1].get('loss', None)
            # 从日志历史记录中获取最新的学习率，如果没有则为 None
            learning_rate = state.log_history[-1].get('learning_rate', None)
            # 使用日志记录器记录当前步数、损失值和学习率
            logger.info(f"Step {state.global_step}: Loss = {loss}, Learning Rate = {learning_rate}")

def get_callbacks(training_args, model_args, script_args):
    """
    返回训练期间要使用的回调函数列表。
    目前，仅包含 LoggingCallback。你可以扩展此函数以添加更多回调。
    """
    # 实例化 LoggingCallback 类，将其添加到回调列表中
    callbacks = [LoggingCallback()]
    # 返回回调列表
    return callbacks

In [20]:
# 获取奖励函数和回调函数

# 调用 get_reward_functions 函数，传入之前实例化的脚本参数对象 script_args
# 该函数会根据脚本参数中指定的奖励函数名称，从奖励函数注册表中获取对应的奖励函数
# 并将这些奖励函数组成一个列表返回，赋值给 reward_functions 变量
reward_functions = get_reward_functions(script_args)

# 调用 get_callbacks 函数，传入训练参数对象 training_args、模型参数对象 model_args 和脚本参数对象 script_args
# 该函数目前仅返回一个包含 LoggingCallback 实例的列表，这个回调函数用于在特定训练步骤记录训练信息
# 将返回的回调函数列表赋值给 callbacks 变量
callbacks = get_callbacks(training_args, model_args, script_args)